In [1]:
import os
print(os.listdir("/kaggle/input/datasets/susandilinn/ait-lectures"))

['Artificial Intelligence Technology_lecture 1.pdf', 'Artificial Intelligence Technology_lecture 6.pdf', 'Artificial Intelligence Technology_lecture 2.pdf', 'Artificial Intelligence Technology_lecture 3.pdf', 'Artificial-Intelligence-Technology_lecture-4.pdf', 'Artificial Intelligence Technology_lecture 7.pdf', 'Artificial Intelligence Technology_lecture 8.pdf', 'Artificial Intelligence Technology_lecture 5.pdf', 'Artificial Intelligence Technology_lecture 9.pdf']


In [2]:
# INSTALLS
!pip install pdfplumber transformers scikit-learn joblib sentencepiece -q

import os
import re
import glob
import torch
import pdfplumber
import joblib
import numpy as np

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans


# DEVICE
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# LOAD MODELS
bert_path = "/kaggle/input/datasets/susandilinn/models/my_bert_model/my_bert_model"

tokenizer = AutoTokenizer.from_pretrained(bert_path)
bert_model = AutoModelForSequenceClassification.from_pretrained(bert_path)

bert_model.to(device)
bert_model.eval()

logistic_model = joblib.load("/kaggle/input/datasets/susandilinn/models/logistic_model.pkl")
tfidf_vectorizer = joblib.load("/kaggle/input/datasets/susandilinn/models/tfidf_vectorizer.pkl")


# PDF TEXT EXTRACTION
def extract_text_from_pdf(pdf_path):

    text = ""

    with pdfplumber.open(pdf_path) as pdf:

        for page in pdf.pages:

            page_text = page.extract_text()

            if page_text:
                text += page_text + "\n"

    return text


# BETTER SENTENCE SPLITTING
def split_into_sentences(text):

    text = text.replace("\n", " ")

    sentences = re.split(r'(?<=[.!?])\s+', text)

    cleaned = []

    for s in sentences:

        s = s.strip()

        if len(s) < 40:
            continue

        if re.match(r"^\d+\.?$", s):
            continue

        cleaned.append(s)

    return cleaned


# REMOVE SLIDE NOISE
def remove_slide_noise(sentences):

    cleaned = []

    noise_patterns = [
        r"cpts\s*\d+",
        r"/\s*\d+",
        r"lecture\s*\d+",
        r"slide\s*\d+",
        r"\b\d{4}\b",
    ]

    for s in sentences:

        s_lower = s.lower()

        if any(re.search(p, s_lower) for p in noise_patterns):
            continue

        cleaned.append(s)

    return cleaned


# REMOVE NUMBERING ARTIFACTS
def remove_numbering_noise(sentences):

    cleaned = []

    for s in sentences:

        s = re.sub(r"\b\d+\.\s*", "", s)

        if re.match(r"^\d+$", s.strip()):
            continue

        if len(s.strip()) < 20:
            continue

        cleaned.append(s.strip())

    return cleaned


# REMOVE URL / REFERENCES
def remove_urls_and_refs(sentences):

    cleaned = []

    for s in sentences:

        s = re.sub(r"http\S+", "", s)
        s = re.sub(r"www\.\S+", "", s)

        if re.search(r"\bref\s*:", s.lower()):
            continue

        cleaned.append(s.strip())

    return cleaned


# REMOVE BROKEN UNICODE
def remove_broken_unicode(sentences):

    cleaned = []

    for s in sentences:

        s = s.encode("ascii", "ignore").decode()

        cleaned.append(s.strip())

    return cleaned


# REMOVE SYMBOL HEAVY LINES
def remove_symbol_heavy_lines(sentences, symbol_threshold=0.25):

    cleaned = []

    for s in sentences:

        non_letters = len(re.findall(r"[^a-zA-Z\s]", s))

        ratio = non_letters / max(1, len(s))

        if ratio > symbol_threshold:
            continue

        cleaned.append(s)

    return cleaned


# REMOVE EQUATION GARBAGE
def remove_equation_noise(sentences):

    cleaned = []

    for s in sentences:

        if re.search(r"[=+\-*/]{3,}", s):
            continue

        cleaned.append(s)

    return cleaned


# TF-IDF FILTER
def tfidf_filter(sentences, keep_ratio=0.6):

    if len(sentences) <= 2:
        return sentences

    vectors = tfidf_vectorizer.transform(sentences)

    scores = np.asarray(vectors.mean(axis=1)).flatten()

    ranked = sorted(zip(sentences, scores), key=lambda x: x[1], reverse=True)

    top_k = int(len(ranked) * keep_ratio)

    return [s[0] for s in ranked[:top_k]]


# BERT SCORING
def bert_score_sentences(sentences, batch_size=16):

    scores = []

    for i in range(0, len(sentences), batch_size):

        batch = sentences[i:i+batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128
        ).to(device)

        with torch.no_grad():
            outputs = bert_model(**inputs)

        probs = torch.softmax(outputs.logits, dim=1)

        for j, sentence in enumerate(batch):

            scores.append((sentence, probs[j][1].item()))

    return scores


# SELECT TOP SENTENCES
def select_top_sentences(scored_sentences, ratio=0.35):

    sorted_by_score = sorted(scored_sentences, key=lambda x: x[1], reverse=True)

    top_k = max(1, int(len(sorted_by_score) * ratio))

    selected = sorted_by_score[:top_k]

    selected_sentences = [s[0] for s in selected]

    original_order = [s[0] for s in scored_sentences]

    selected_sentences.sort(key=lambda x: original_order.index(x))

    return selected_sentences


# REMOVE REDUNDANCY
def remove_redundancy(sentences, threshold=0.8):

    if len(sentences) <= 1:
        return sentences

    vectors = TfidfVectorizer().fit_transform(sentences)

    similarity = cosine_similarity(vectors)

    filtered = []
    used = set()

    for i in range(len(sentences)):

        if i in used:
            continue

        filtered.append(sentences[i])

        for j in range(i+1, len(sentences)):

            if similarity[i][j] > threshold:
                used.add(j)

    return filtered


# COMPRESS SENTENCES
def compress_sentence(sentence):

    sentence = sentence.strip()

    words = sentence.split()

    new_words = []

    for w in words:
        if len(new_words) == 0 or new_words[-1].lower() != w.lower():
            new_words.append(w)

    sentence = " ".join(new_words)

    if len(sentence) > 180:
        sentence = sentence.split(".")[0] + "."

    return sentence


# GENERATE TITLES
def generate_dynamic_titles(sentences, num_sections=2):

    if len(sentences) < num_sections:
        return [("KEY CONCEPTS", sentences)]

    vectorizer = TfidfVectorizer(stop_words="english")

    X = vectorizer.fit_transform(sentences)

    kmeans = KMeans(n_clusters=num_sections, random_state=42, n_init=10)

    labels = kmeans.fit_predict(X)

    feature_names = vectorizer.get_feature_names_out()

    sections = {}

    for i, label in enumerate(labels):
        sections.setdefault(label, []).append(sentences[i])

    results = []

    for label, sents in sections.items():

        cluster_vec = vectorizer.transform(sents).mean(axis=0)

        top_indices = cluster_vec.A1.argsort()[-3:]

        title_words = [feature_names[i] for i in top_indices]

        title = " ".join(title_words).upper()

        results.append((title, sents))

    return results


# HYBRID SUMMARIZER
def summarize_lecture_hybrid(pdf_path, final_ratio=0.35):

    raw_text = extract_text_from_pdf(pdf_path)

    sentences = split_into_sentences(raw_text)

    sentences = remove_slide_noise(sentences)
    sentences = remove_numbering_noise(sentences)
    sentences = remove_urls_and_refs(sentences)
    sentences = remove_broken_unicode(sentences)
    sentences = remove_symbol_heavy_lines(sentences)
    sentences = remove_equation_noise(sentences)

    if len(sentences) == 0:
        return None

    filtered = tfidf_filter(sentences, keep_ratio=0.6)

    scored = bert_score_sentences(filtered)

    selected = select_top_sentences(scored, ratio=final_ratio)

    final_sentences = remove_redundancy(selected)

    compressed = [compress_sentence(s) for s in final_sentences]

    sections = generate_dynamic_titles(compressed, num_sections=2)

    formatted_summary = ""

    for title, sents in sections:

        formatted_summary += f"\nTITLE: {title}\n"

        for s in sents:

            formatted_summary += f"- {s}\n"

    return formatted_summary


# PROCESS ALL LECTURES
output_file_path = "/kaggle/working/all_summaries.txt"

with open(output_file_path, "w", encoding="utf-8") as f:

    for slide_path in glob.glob("/kaggle/input/datasets/susandilinn/ait-lectures/*.pdf"):

        summary = summarize_lecture_hybrid(slide_path, final_ratio=0.35)

        if summary:

            file_name = os.path.basename(slide_path)

            print(f"\nFILE: {file_name}")
            print(summary)

            f.write(f"\nFILE: {file_name}\n")
            f.write(summary)
            f.write("\n")


print("Cheatsheet TXT file created successfully!")
print(f"File location: {output_file_path}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 84.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 105.3 MB/s eta 0:00:0000:01


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


FILE: Artificial Intelligence Technology_lecture 1.pdf

TITLE: UNSUPERVISED DATA LEARNING
- Unsupervised learning models automatically extract features and find patterns in the data Reinforcement Learning Reinforcement learning trains an algorithm with a reward system, providing feedback when an artificial intelligence agent performs the best action in a particular situation.
- Machine Learning Algorithms Classification is applied when output has finite and discrete values Linear Regression Linear Regression: Model Representation Linear regression is a linear approach for modelling the relationship between dependent and independent variables.
- Machine Learning Algorithms Regression is applied when output has continuous values Represents relationship between input (x) and output (y) variables Output variable (y) is a dependent variable, also called target variable, or response.
- Through methods such as classification, regression, and prediction, supervised learning uses models to pre